# Stage 4 — Unsupervised anomaly maps from reconstruction error

No labels, no fine-tuning: the pretrained MAE reconstructs held-out slices;
brain-masked |x − x̂| is the anomaly score. Subject score = top-25 % slice mean.

Checks: AUC (demented vs CDR0) on TEST, Spearman(score, nWBV) expected
**negative**, Spearman(score, age) positive-but-weaker, plus example heatmaps.
Budget: ~20–60 min GPU.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_KAGGLE = Path("/kaggle").is_dir()
if IN_KAGGLE:
    repo = Path("mv-brainmae")
    if not repo.is_dir():
        # set your repo URL after pushing to GitHub
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/YOUR_GITHUB_USERNAME/mv-brainmae.git"],
                       check=True)
else:
    repo = Path("..")  # local: notebooks/ inside a clone
sys.path.insert(0, str(repo.resolve()))
os.chdir(repo)
print("repo:", repo.resolve())

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from mvbrainmae.utils import load_config
from mvbrainmae.anomaly import run_anomaly

cfg = load_config("configs/anomaly.yaml")
metrics = run_anomaly(cfg)
print(metrics)

In [ ]:
# ---- VERIFICATION ----
import json
from PIL import Image
import matplotlib.pyplot as plt
m = json.load(open("results/anomaly_metrics.json"))
for arm, v in m.items():
    print(f"{arm}: AUC_topk={v['auc_topk']:.3f}  spearman(nWBV)={v['spearman_nwbv']:+.3f}"
          f" (p={v['spearman_nwbv_p']:.3g})")
for f in sorted(Path("figures").glob("anomaly_examples_*.png")):
    fig, ax = plt.subplots(figsize=(8, 6)); ax.imshow(Image.open(f)); ax.axis("off")
    plt.tight_layout(); plt.show()